# Análise exploratória da base de varejo

Execute as células de cima para baixo. A análise usa somente **pandas** e **matplotlib**.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PASTA_RESULTADOS = Path('resultados')
PASTA_RESULTADOS.mkdir(exist_ok=True)

## 1. Ler e conhecer a base

O arquivo usa ponto e vírgula como separador.

In [ ]:
df = pd.read_csv('varejo.csv', sep=';')

print('Registros e colunas:', df.shape)
print('\nTipos de dados:')
print(df.dtypes)
df.head()

## 2. Verificar a qualidade

Vamos procurar nulos, duplicatas, colunas vazias, datas inválidas e categorias estranhas.

In [ ]:
print('Nulos por coluna:')
print(df.isna().sum())
print('\nLinhas totalmente duplicadas:', df.duplicated().sum())

colunas_vazias = df.columns[df.isna().all()].tolist()
print('Colunas totalmente vazias:', colunas_vazias)

datas_teste = pd.to_datetime(df['DATA'], format='%d/%m/%Y', errors='coerce')
print('Datas inválidas:', datas_teste.isna().sum())
print('\nCategorias:')
print(df['PR_CAT'].value_counts(dropna=False))

## 3. Limpar a base

Removemos colunas vazias, transformamos `#N/D` em `Sem Categoria`, retiramos apenas duplicatas completas e convertemos a data. Repetições de `CO_ID` são mantidas, pois uma compra pode ter vários produtos.

In [ ]:
df = df.dropna(axis=1, how='all')
df['PR_CAT'] = df['PR_CAT'].fillna('Sem Categoria')
df['PR_CAT'] = df['PR_CAT'].str.strip().replace('#N/D', 'Sem Categoria')

linhas_antes = len(df)
df = df.drop_duplicates().copy()
df['DATA'] = pd.to_datetime(df['DATA'], format='%d/%m/%Y', errors='coerce')

print('Duplicatas removidas:', linhas_antes - len(df))
print('Registros após a limpeza:', len(df))
print('Datas inválidas após a conversão:', df['DATA'].isna().sum())
df.to_csv(PASTA_RESULTADOS / 'varejo_limpo.csv', index=False, encoding='utf-8-sig')

## 4. Estatísticas do número de filhos

A coluna `CL_FHL` representa o número de filhos do cliente.

In [ ]:
filhos = df['CL_FHL']
estatisticas_filhos = pd.Series({
    'contagem': filhos.count(),
    'média': filhos.mean(),
    'mediana': filhos.median(),
    'desvio padrão': filhos.std(),
    'moda': filhos.mode().iloc[0],
    'mínimo': filhos.min(),
    '1º quartil': filhos.quantile(0.25),
    '2º quartil': filhos.quantile(0.50),
    '3º quartil': filhos.quantile(0.75),
    'máximo': filhos.max(),
})
estatisticas_filhos.round(2)

## 5. Agrupamentos e gráficos

Para contar compras, deixamos uma linha por `CO_ID`. Para contar itens por categoria, usamos todas as linhas da base limpa.

In [ ]:
compras = df.drop_duplicates(subset='CO_ID').copy()
compras_por_genero = compras.groupby('CL_GENERO')['CO_ID'].nunique().sort_values(ascending=False)
itens_por_categoria = df.groupby('PR_CAT').size().sort_values(ascending=False)
compras_por_mes = compras.groupby(compras['DATA'].dt.to_period('M'))['CO_ID'].nunique()

print('Compras únicas por gênero:')
print(compras_por_genero)
print('\nItens por categoria:')
print(itens_por_categoria)
print('\nCompras por mês:')
print(compras_por_mes)

In [ ]:
plt.figure(figsize=(8, 4))
compras_por_genero.plot(kind='bar', color=['#4C78A8', '#F58518'])
plt.title('Compras únicas por gênero')
plt.xlabel('Gênero')
plt.ylabel('Quantidade de compras')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(PASTA_RESULTADOS / 'compras_por_genero.png', dpi=150)
plt.show()

plt.figure(figsize=(9, 4))
itens_por_categoria.plot(kind='bar', color='#54A24B')
plt.title('Quantidade de itens por categoria')
plt.xlabel('Categoria')
plt.ylabel('Quantidade de itens')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.savefig(PASTA_RESULTADOS / 'itens_por_categoria.png', dpi=150)
plt.show()

## 6. Conclusões

- Foram removidas linhas totalmente duplicadas.
- `ALIMENTOS` é a categoria com mais itens registrados.
- Há mais compras únicas de clientes do gênero `F`.
- A moda e a mediana de `CL_FHL` são 0.
- `CO_ID` repetido é esperado, pois uma compra pode conter diversos produtos.
- Não havia nulos nem datas inválidas; `#N/D` foi padronizado como `Sem Categoria`.